In [2]:
import pandas as pd

# Load original dataset
df = pd.read_csv("../data/perishable_goods_management.csv")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate records:", df.duplicated().sum())

print("\nNumerical column summary:")
display(df.describe().T)

Dataset shape: (100000, 42)

Column names:
['record_id', 'product_id', 'product_name', 'category', 'store_id', 'region', 'supplier_id', 'transaction_date', 'expiration_date', 'shelf_life_days', 'days_remaining_at_purchase', 'storage_temp', 'temp_deviation', 'base_price', 'cost_price', 'initial_quantity', 'spoilage_sensitivity', 'day_of_week', 'is_weekend', 'month', 'daily_demand', 'demand_variability', 'temp_abuse_events', 'distribution_hours', 'handling_score', 'packaging_score', 'spoilage_risk', 'was_spoiled', 'quality_grade', 'days_until_expiry', 'markdown_applied', 'discount_pct', 'selling_price', 'units_sold', 'units_wasted', 'waste_pct', 'revenue', 'waste_cost', 'profit', 'profit_margin_pct', 'supplier_score', 'is_promoted']

Data types:
record_id                       int64
product_id                        str
product_name                      str
category                          str
store_id                          str
region                            str
supplier_id       

,count,mean,std,min,25%,50%,75%,max
record_id,100000.0,50000.500000,28867.657797,1.000,25000.750,50000.500,75000.2500,100000.000
shelf_life_days,100000.0,67.169300,140.550322,1.000,5.000,9.000,24.0000,730.000
days_remaining_at_purchase,100000.0,56.114940,118.160892,0.000,4.000,8.000,20.0000,723.000
storage_temp,100000.0,3.127054,9.503199,-27.700,1.100,4.000,6.6000,27.700
temp_deviation,100000.0,1.597402,1.206793,0.000,0.600,1.400,2.3000,9.000
base_price,100000.0,36.408563,87.424777,1.000,6.270,10.220,17.0100,499.980
cost_price,100000.0,20.058530,48.893810,0.410,3.360,5.580,9.5000,346.610
initial_quantity,100000.0,254.851800,141.686286,10.000,132.000,255.000,377.0000,500.000
spoilage_sensitivity,100000.0,0.685802,0.215392,0.300,0.500,0.750,0.9000,0.950
day_of_week,100000.0,3.001080,2.000745,0.000,1.000,3.000,5.0000,6.000


In [3]:
# 1. Convert date columns
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"], errors="coerce"
)

df["expiration_date"] = pd.to_datetime(
    df["expiration_date"], errors="coerce"
)

# 2. Check invalid dates
print("Invalid transaction dates:",
      df["transaction_date"].isna().sum())

print("Invalid expiration dates:",
      df["expiration_date"].isna().sum())

# 3. Check negative values in selected columns
columns_to_check = [
    "daily_demand",
    "initial_quantity",
    "units_sold",
    "units_wasted",
    "base_price",
    "cost_price",
    "selling_price",
    "shelf_life_days",
    "days_until_expiry"
]

print("\nNegative values:")
for col in columns_to_check:
    print(col, ":", (df[col] < 0).sum())

# 4. Check binary columns
binary_columns = [
    "is_weekend",
    "was_spoiled",
    "markdown_applied",
    "is_promoted"
]

print("\nBinary column validation:")
for col in binary_columns:
    print(col, ":", sorted(df[col].unique()))

# 5. Check business relationships
print("\nUnits sold greater than initial quantity:",
      (df["units_sold"] > df["initial_quantity"]).sum())

print("Units wasted greater than initial quantity:",
      (df["units_wasted"] > df["initial_quantity"]).sum())

print("Expiration before transaction:",
      (df["expiration_date"] < df["transaction_date"]).sum())

C:\Users\Sriram S\AppData\Local\Temp\ipykernel_51480\2466430815.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["transaction_date"] = pd.to_datetime(
C:\Users\Sriram S\AppData\Local\Temp\ipykernel_51480\2466430815.py:6: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["expiration_date"] = pd.to_datetime(


Invalid transaction dates: 0
Invalid expiration dates: 0

Negative values:
daily_demand : 0
initial_quantity : 0
units_sold : 0
units_wasted : 0
base_price : 0
cost_price : 0
selling_price : 0
shelf_life_days : 0
days_until_expiry : 0

Binary column validation:
is_weekend : [np.int64(0), np.int64(1)]
was_spoiled : [np.int64(0), np.int64(1)]
markdown_applied : [np.int64(0), np.int64(1)]
is_promoted : [np.int64(0), np.int64(1)]

Units sold greater than initial quantity: 0
Units wasted greater than initial quantity: 0
Expiration before transaction: 0


In [4]:
# Check expiry-related consistency

print("Days until expiry vs shelf life:")
print(
    "Days until expiry greater than shelf life:",
    (df["days_until_expiry"] > df["shelf_life_days"]).sum()
)

print(
    "Days remaining at purchase greater than shelf life:",
    (df["days_remaining_at_purchase"] > df["shelf_life_days"]).sum()
)

# Check date difference
date_difference = (
    df["expiration_date"] - df["transaction_date"]
).dt.days

print(
    "\nExpiration date difference mismatch:",
    (date_difference != df["days_until_expiry"]).sum()
)

# Check waste percentage range
print(
    "\nWaste percentage outside 0-100:",
    ((df["waste_pct"] < 0) | (df["waste_pct"] > 100)).sum()
)

# Check discount percentage range
print(
    "Discount percentage outside 0-100:",
    ((df["discount_pct"] < 0) | (df["discount_pct"] > 100)).sum()
)

Days until expiry vs shelf life:
Days until expiry greater than shelf life: 0
Days remaining at purchase greater than shelf life: 0

Expiration date difference mismatch: 74191

Waste percentage outside 0-100: 0
Discount percentage outside 0-100: 0


In [5]:
date_difference = (
    df["expiration_date"] - df["transaction_date"]
).dt.days

mismatch = df[
    date_difference != df["days_until_expiry"]
].copy()

mismatch["calculated_days"] = date_difference[
    date_difference != df["days_until_expiry"]
]

display(
    mismatch[
        [
            "transaction_date",
            "expiration_date",
            "shelf_life_days",
            "days_remaining_at_purchase",
            "days_until_expiry",
            "calculated_days"
        ]
    ].head(10)
)

,transaction_date,expiration_date,shelf_life_days,days_remaining_at_purchase,days_until_expiry,calculated_days
0,2024-09-25,2024-09-29,4,4,2,4
1,2023-04-14,2023-04-21,9,7,4,7
2,2024-10-25,2024-10-27,2,2,0,2
3,2023-11-29,2024-02-17,87,80,79,80
5,2024-08-21,2025-07-06,468,319,317,319
6,2023-02-18,2023-02-27,10,9,8,9
7,2024-04-24,2025-02-16,330,298,297,298
9,2023-07-02,2023-07-08,6,6,5,6
10,2023-06-15,2023-06-19,6,4,3,4
13,2023-07-25,2024-05-18,349,298,295,298


In [6]:
print("Mismatch count:", len(mismatch))

print("\nDays until expiry:")
print(mismatch["days_until_expiry"].describe())

print("\nCalculated days:")
print(mismatch["calculated_days"].describe())

Mismatch count: 74191

Days until expiry:
count    74191.000000
mean        54.842124
std        118.830363
min          0.000000
25%          2.000000
50%          6.000000
75%         18.000000
max        720.000000
Name: days_until_expiry, dtype: float64

Calculated days:
count    74191.000000
mean        56.782723
std        118.854947
min          1.000000
25%          4.000000
50%          8.000000
75%         20.000000
max        723.000000
Name: calculated_days, dtype: float64


In [7]:
print(
    "Days remaining at purchase matches date difference:",
    (df["days_remaining_at_purchase"] == date_difference).sum()
)

print(
    "Days until expiry matches days remaining at purchase:",
    (df["days_until_expiry"] ==
     df["days_remaining_at_purchase"]).sum()
)

print(
    "Days until expiry matches date difference:",
    (df["days_until_expiry"] == date_difference).sum()
)

Days remaining at purchase matches date difference: 100000
Days until expiry matches days remaining at purchase: 25809
Days until expiry matches date difference: 25809


In [8]:
numeric_df = df.select_dtypes(include="number")

for col in numeric_df.columns:
    Q1 = numeric_df[col].quantile(0.25)
    Q3 = numeric_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((numeric_df[col] < lower) |
                (numeric_df[col] > upper)).sum()

    if outliers > 0:
        print(f"{col}: {outliers} potential outliers")

shelf_life_days: 19685 potential outliers
days_remaining_at_purchase: 19620 potential outliers
storage_temp: 19915 potential outliers
temp_deviation: 1554 potential outliers
base_price: 12968 potential outliers
cost_price: 12495 potential outliers
daily_demand: 6530 potential outliers
temp_abuse_events: 7838 potential outliers
spoilage_risk: 647 potential outliers
was_spoiled: 19442 potential outliers
days_until_expiry: 19604 potential outliers
discount_pct: 7901 potential outliers
selling_price: 12206 potential outliers
units_wasted: 5184 potential outliers
waste_pct: 8003 potential outliers
revenue: 9753 potential outliers
waste_cost: 12082 potential outliers
profit: 15756 potential outliers
profit_margin_pct: 6031 potential outliers
is_promoted: 14898 potential outliers


In [9]:
check_columns = [
    "storage_temp",
    "temp_deviation",
    "daily_demand",
    "base_price",
    "cost_price",
    "units_wasted",
    "waste_pct",
    "spoilage_risk",
    "profit",
    "profit_margin_pct"
]

for col in check_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    print(f"\n{col}")
    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())
    print("Lower IQR limit:", round(lower, 2))
    print("Upper IQR limit:", round(upper, 2))


storage_temp
Minimum: -27.7
Maximum: 27.7
Lower IQR limit: -7.15
Upper IQR limit: 14.85

temp_deviation
Minimum: 0.0
Maximum: 9.0
Lower IQR limit: -1.95
Upper IQR limit: 4.85

daily_demand
Minimum: 0
Maximum: 1118
Lower IQR limit: -112.5
Upper IQR limit: 203.5

base_price
Minimum: 1.0
Maximum: 499.98
Lower IQR limit: -9.84
Upper IQR limit: 33.12

cost_price
Minimum: 0.41
Maximum: 346.61
Lower IQR limit: -5.85
Upper IQR limit: 18.71

units_wasted
Minimum: 0
Maximum: 500
Lower IQR limit: -120.0
Upper IQR limit: 200.0

waste_pct
Minimum: 0.0
Maximum: 100.0
Lower IQR limit: -46.2
Upper IQR limit: 77.0

spoilage_risk
Minimum: 0.053
Maximum: 0.469
Lower IQR limit: 0.05
Upper IQR limit: 0.33

profit
Minimum: -169495.0
Maximum: 136195.21
Lower IQR limit: -1610.17
Upper IQR limit: 2554.38

profit_margin_pct
Minimum: -1798.8
Maximum: 60.3
Lower IQR limit: -61.65
Upper IQR limit: 102.75


In [10]:
# Highest demand
print("TOP 10 DAILY DEMAND")
display(
    df.nlargest(10, "daily_demand")[
        ["product_name", "category", "daily_demand",
         "initial_quantity", "units_sold", "units_wasted"]
    ]
)

# Largest losses
print("\nTOP 10 NEGATIVE PROFITS")
display(
    df.nsmallest(10, "profit")[
        ["product_name", "category", "initial_quantity",
         "units_sold", "units_wasted", "revenue",
         "waste_cost", "profit"]
    ]
)

# Extreme storage temperatures
print("\nEXTREME STORAGE TEMPERATURES")
display(
    pd.concat([
        df.nsmallest(5, "storage_temp"),
        df.nlargest(5, "storage_temp")
    ])[
        ["product_name", "category", "storage_temp",
         "temp_deviation", "spoilage_risk"]
    ]
)

TOP 10 DAILY DEMAND


,product_name,category,daily_demand,initial_quantity,units_sold,units_wasted
81293,Salads,Ready_to_Eat,1118,500,500,0
62406,Salads,Ready_to_Eat,1096,488,488,0
17093,Sushi,Ready_to_Eat,1077,488,488,0
63304,Salads,Ready_to_Eat,1073,469,328,141
82750,Sandwiches,Ready_to_Eat,1065,500,500,0
83864,Wraps,Ready_to_Eat,1055,456,319,137
86634,Sandwiches,Ready_to_Eat,1055,453,453,0
63955,Soups,Ready_to_Eat,1038,476,476,0
54353,Soups,Ready_to_Eat,1036,479,479,0
89856,Salads,Ready_to_Eat,1029,492,492,0



TOP 10 NEGATIVE PROFITS


,product_name,category,initial_quantity,units_sold,units_wasted,revenue,waste_cost,profit
51746,Eye Drops,Pharmaceuticals,500,0,500,0.0,169495.00,-169495.00
13635,Probiotics,Pharmaceuticals,496,0,496,0.0,166388.16,-166388.16
11141,Vaccines,Pharmaceuticals,487,0,487,0.0,154281.60,-154281.60
2427,Probiotics,Pharmaceuticals,499,0,499,0.0,152893.60,-152893.60
20526,Eye Drops,Pharmaceuticals,499,0,499,0.0,152873.64,-152873.64
39007,Insulin,Pharmaceuticals,472,0,472,0.0,145975.44,-145975.44
41808,Insulin,Pharmaceuticals,481,0,481,0.0,145670.85,-145670.85
99501,Vaccines,Pharmaceuticals,467,0,467,0.0,143714.58,-143714.58
97616,Eye Drops,Pharmaceuticals,495,0,495,0.0,143386.65,-143386.65
70536,Biologics,Pharmaceuticals,458,0,458,0.0,141865.50,-141865.50



EXTREME STORAGE TEMPERATURES


,product_name,category,storage_temp,temp_deviation,spoilage_risk
19674,Ice Cream,Frozen_Meals,-27.7,7.7,0.202
93640,Ice Cream,Frozen_Meals,-27.4,7.4,0.141
28790,Pizza,Frozen_Meals,-26.9,6.9,0.198
11180,Frozen Vegetables,Frozen_Meals,-26.8,6.8,0.160
30747,Frozen Desserts,Frozen_Meals,-26.7,6.7,0.202
209,Cakes,Bakery,27.7,7.7,0.334
52813,Bread,Bakery,27.5,7.5,0.205
89937,Donuts,Bakery,27.4,7.4,0.209
32820,Donuts,Bakery,27.2,7.2,0.263
57160,Bagels,Bakery,26.9,6.9,0.211


In [11]:
# Check whether profit = revenue - waste_cost
profit_difference = (
    df["profit"] - (df["revenue"] - df["waste_cost"])
)

print("Profit formula mismatches:",
      (profit_difference.abs() > 0.01).sum())

# Inspect the largest mismatches
display(
    df.loc[
        profit_difference.abs().nlargest(10).index,
        [
            "revenue",
            "waste_cost",
            "profit"
        ]
    ].assign(
        calculated_profit=lambda x:
        x["revenue"] - x["waste_cost"]
    )
)

Profit formula mismatches: 92348


,revenue,waste_cost,profit,calculated_profit
18530,199767.15,0.0,29902.95,199767.15
18280,236062.40,0.0,74176.20,236062.40
17786,231338.40,0.0,72678.24,231338.40
99585,230079.10,0.0,72318.90,230079.10
93648,226954.68,0.0,72260.04,226954.68
95330,238915.62,0.0,85941.75,238915.62
68715,238721.85,0.0,86228.15,238721.85
54729,226835.05,0.0,77245.16,226835.05
74954,233370.28,0.0,84622.56,233370.28
62360,184883.76,0.0,40221.00,184883.76


In [12]:
# Calculate possible profit components
df["estimated_purchase_cost"] = (
    df["units_sold"] * df["cost_price"]
)

df["estimated_total_cost"] = (
    df["estimated_purchase_cost"] + df["waste_cost"]
)

df["calculated_profit_v2"] = (
    df["revenue"] - df["estimated_total_cost"]
)

# Compare with stored profit
difference = (
    df["profit"] - df["calculated_profit_v2"]
)

print(
    "Mismatches using revenue - purchase cost - waste cost:",
    (difference.abs() > 0.01).sum()
)

print("\nSample comparison:")

display(
    df[[
        "revenue",
        "units_sold",
        "cost_price",
        "waste_cost",
        "profit",
        "calculated_profit_v2"
    ]].head(10)
)

Mismatches using revenue - purchase cost - waste cost: 0

Sample comparison:


,revenue,units_sold,cost_price,waste_cost,profit,calculated_profit_v2
0,358.80,138,1.22,24.40,166.04,166.04
1,2070.75,251,4.18,426.36,595.21,595.21
2,618.24,483,2.12,0.00,-405.72,-405.72
3,99960.12,477,87.99,0.00,57988.89,57988.89
4,2064.48,391,3.03,0.00,879.75,879.75
5,0.00,0,84.87,22914.90,-22914.90,-22914.90
6,341.94,41,4.39,4.39,157.56,157.56
7,0.00,0,6.46,2990.98,-2990.98,-2990.98
8,34758.09,379,36.77,0.00,20822.26,20822.26
9,1436.34,111,7.15,0.00,642.69,642.69


In [13]:
import os

# Create output folder if it doesn't exist
os.makedirs("data/processed", exist_ok=True)

# Save cleaned dataset
output_path = "data/processed/cleaned_data.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("File location:", output_path)
print("Dataset shape:", df.shape)

Cleaned dataset saved successfully!
File location: data/processed/cleaned_data.csv
Dataset shape: (100000, 45)


In [14]:
# Remove temporary investigation columns
df.drop(
    columns=[
        "estimated_purchase_cost",
        "estimated_total_cost",
        "calculated_profit_v2"
    ],
    errors="ignore",
    inplace=True
)

# Save the cleaned dataset
df.to_csv("data/processed/cleaned_data.csv", index=False)

print("Saved successfully!")
print("Final shape:", df.shape)

C:\Users\Sriram S\AppData\Local\Temp\ipykernel_51480\3080337784.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(


Saved successfully!
Final shape: (100000, 42)


In [15]:
print("Current DataFrame shape:", df.shape)

print("\nTemporary columns:")
print([
    col for col in df.columns
    if col in [
        "estimated_purchase_cost",
        "estimated_total_cost",
        "calculated_profit_v2"
    ]
])

print("\nTotal columns:", len(df.columns))

Current DataFrame shape: (100000, 42)

Temporary columns:
[]

Total columns: 42


In [16]:
# Remove temporary investigation columns safely
df = df.drop(columns=[
    "estimated_purchase_cost",
    "estimated_total_cost",
    "calculated_profit_v2"
], errors="ignore")

# Save again
df.to_csv("data/processed/cleaned_data.csv", index=False)

print("\nFinal dataset shape:", df.shape)


Final dataset shape: (100000, 42)


In [17]:
df.to_csv("data/processed/cleaned_data.csv", index=False)

# Verify the saved file
check_df = pd.read_csv("data/processed/cleaned_data.csv")

print("Saved dataset shape:", check_df.shape)

Saved dataset shape: (100000, 42)
